In [ ]:
import importlib
import sys

# Force reload the fetch_picDk_place module to get latest code changes
if 'when2plan.envs.fetch_pick_place.env' in sys.modules:
    import when2plan.envs.fetch_pick_place.env
    importlib.reload(when2plan.envs.fetch_pick_place.env)

from notebook_utils import load_hydra_config, to_pil
from when2plan.envs.fetch_pick_place.env import FetchPickPlaceEnv
from omegaconf import OmegaConf
import numpy as np
from IPython.display import display, clear_output
import numpy as np
import torch as th
import time
import copy
from when2plan.utils import collect_render_data_with_planner, get_model_free_metrics, plot_model_free_metrics
from when2plan.models.pretrain_model_free import pretrain_model_free_sub_env
from argparse import Namespace
from when2plan.envs.meta_env import MetaEnv

In [ ]:
in_distribution_cfg = load_hydra_config("../experiments/configs", "config", ["env=fetch_pick_place", 'experiment=fetch_pick_place_pretrain'])
out_of_distribution_cfg = load_hydra_config("../experiments/configs", "config", ["env=fetch_pick_place", 'experiment=fetch_pick_place_pretrain'])

# Set in distribution vs out of distribution parameters
fetch_pick_place_id = FetchPickPlaceEnv(in_distribution_cfg, render_mode="rgb_array")
fetch_pick_place_ood = FetchPickPlaceEnv(out_of_distribution_cfg, render_mode="rgb_array")

fetch_pick_place_id.set_cache_task_range(0, in_distribution_cfg.experiment.num_pretrained_tasks)
fetch_pick_place_ood.set_cache_task_range(0, 20)  


In [ ]:
# Pretrain model free agent on in-distribution environment
args = Namespace(
    sub_env="fetch_pick_place",
    model_free_observation_space="image",
    model_free_action_space="continuous",
    num_actors=4,
    steps=2000,
    batch_size=64,
    epochs=50,
    base_seed=42,
    plot=True,
    save_actor=True,
)
pretrained_networks, env = pretrain_model_free_sub_env(args, fetch_pick_place_id)

In [ ]:
# Create meta environments for in-distribution and out-of-distribution
id_meta_env = MetaEnv(in_distribution_cfg, fetch_pick_place_id, pretrained_networks)
ood_meta_env = MetaEnv(out_of_distribution_cfg, fetch_pick_place_ood, pretrained_networks)

# Get type of meta step from config
model_free_action = 0

# Get metrics
metrics = get_model_free_metrics(id_meta_env, ood_meta_env, n_steps=1000, model_free_action=model_free_action)

# plot metrics
plot_model_free_metrics(metrics, save_path='maze_model_free_metrics.png')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter
from IPython.display import HTML

id_meta_env = MetaEnv(in_distribution_cfg, fetch_pick_place_id, pretrained_networks)

# Render 1 episodes with the pretrained model free agent
num_episodes = 20
all_frames = []
episode_boundaries = []  # Store (start_idx, end_idx, reward, steps) for each episode

for episode_idx in range(num_episodes):
    obs, info = id_meta_env.reset()
    done = False
    total_reward = 0
    step_count = 0
    episode_start = len(all_frames)
    
    while not done:
        model_free_action = 0
        obs, reward, terminated, truncated, info = id_meta_env.step(model_free_action)
        
        # Collect frame instead of displaying it
        frame = id_meta_env.render()
        all_frames.append(frame)
        
        total_reward += reward
        step_count += 1
        
        if terminated or truncated:
            done = True
    
    episode_end = len(all_frames)
    episode_boundaries.append((episode_start, episode_end, total_reward, step_count))
    print(f"Episode {episode_idx + 1}/{num_episodes} - Total Reward: {total_reward:.2f}, Steps: {step_count}, Task idx: {info['task_idx']}")

# Create and display video animation
print(f"\nCreating animation from {len(all_frames)} frames across {num_episodes} episodes...")

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')

im = ax.imshow(all_frames[0])
text = ax.text(
    0.02, 0.98, '', 
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
)

def update(frame_idx):
    im.set_array(all_frames[frame_idx])
    
    # Find which episode this frame belongs to
    for ep_num, (start, end, reward, steps) in enumerate(episode_boundaries):
        if start <= frame_idx < end:
            local_frame = frame_idx - start
            info_text = (
                f"Episode: {ep_num + 1}/{num_episodes}\n"
                f"Frame: {local_frame}/{end - start}\n"
                f"Total Steps: {steps}\n"
                f"Total Reward: {reward:.2f}"
            )
            text.set_text(info_text)
            break
    
    return [im, text]

anim = animation.FuncAnimation(
    fig, update, 
    frames=len(all_frames),
    interval=50,  # 50ms between frames (20 fps)
    blit=True
)

plt.close()
display(HTML(anim.to_html5_video()))

# Clear rendering 
id_meta_env.close()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter
from IPython.display import HTML

id_meta_env = MetaEnv(out_of_distribution_cfg, fetch_pick_place_ood, pretrained_networks)

# Render 1 episodes with the pretrained model free agent
num_episodes = 20
all_frames = []
episode_boundaries = []  # Store (start_idx, end_idx, reward, steps) for each episode

for episode_idx in range(num_episodes):
    obs, info = id_meta_env.reset()
    done = False
    total_reward = 0
    step_count = 0
    episode_start = len(all_frames)
    
    while not done:
        model_free_action = 0
        obs, reward, terminated, truncated, info = id_meta_env.step(model_free_action)
        
        # Collect frame instead of displaying it
        frame = id_meta_env.render()
        all_frames.append(frame)
        
        total_reward += reward
        step_count += 1
        
        if terminated or truncated:
            done = True
    
    episode_end = len(all_frames)
    episode_boundaries.append((episode_start, episode_end, total_reward, step_count))
    print(f"Episode {episode_idx + 1}/{num_episodes} - Total Reward: {total_reward:.2f}, Steps: {step_count}, Task idx: {info['task_idx']}")

# Create and display video animation
print(f"\nCreating animation from {len(all_frames)} frames across {num_episodes} episodes...")

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')

im = ax.imshow(all_frames[0])
text = ax.text(
    0.02, 0.98, '', 
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
)

def update(frame_idx):
    im.set_array(all_frames[frame_idx])
    
    # Find which episode this frame belongs to
    for ep_num, (start, end, reward, steps) in enumerate(episode_boundaries):
        if start <= frame_idx < end:
            local_frame = frame_idx - start
            info_text = (
                f"Episode: {ep_num + 1}/{num_episodes}\n"
                f"Frame: {local_frame}/{end - start}\n"
                f"Total Steps: {steps}\n"
                f"Total Reward: {reward:.2f}"
            )
            text.set_text(info_text)
            break
    
    return [im, text]

anim = animation.FuncAnimation(
    fig, update, 
    frames=len(all_frames),
    interval=50,  # 50ms between frames (20 fps)
    blit=True
)

plt.close()
display(HTML(anim.to_html5_video()))

# Clear rendering 
id_meta_env.close()